In [ ]:
!nvidia-smi

Mon Sep 21 17:06:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers accelerate datasets

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
import pandas as pd

df = pd.read_csv("/content/gsm8k_200_questions.csv")

print("Number of questions:", len(df))

df[["id", "question", "correct_answer"]].head()

Number of questions: 200


,id,question,correct_answer
0,1,Darrell and Allen's ages are in the ratio of 7...,109
1,2,Lorraine and Colleen are trading stickers for ...,89
2,3,Indras has 6 letters in her name. Her sister's...,13
3,4,Bethany can run 10 laps on the track in one ho...,5
4,5,An ice cream truck is traveling through a neig...,25


In [ ]:
# ==========================================
# DIRECT PROMPTING - TEST QUESTION 1
# ==========================================

import torch

# Get the first GSM8K question
question = df.iloc[0]["question"]
correct_answer = df.iloc[0]["correct_answer"]

# Direct Prompt
# We specifically do NOT ask the model to reason step-by-step
messages = [
    {
        "role": "user",
        "content": f"""Solve the following math word problem.

Do not provide an explanation or show your working.
Return only the final numerical answer in this exact format:

Final Answer: <number>

Problem:
{question}"""
    }
]

# Format the message correctly for Qwen
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Convert the text into tokens and send them to the GPU
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Generate the model's answer
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Remove the original prompt from the generated output
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

# Convert the generated tokens back into readable text
model_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

# Show the results
print("QUESTION:")
print(question)

print("\nMODEL RESPONSE:")
print(model_response)

print("\nCORRECT GSM8K ANSWER:")
print(correct_answer)

QUESTION:
Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now.

MODEL RESPONSE:
Final Answer: 82

CORRECT GSM8K ANSWER:
109


In [ ]:
# ==========================================
# DIRECT PROMPTING - TEST QUESTIONS 1 TO 5
# ==========================================

import torch
import re
import pandas as pd

results = []

# Run only the first 5 questions
for index, row in df.head(5).iterrows():

    question_id = row["id"]
    question = row["question"]
    correct_answer = str(row["correct_answer"]).replace(",", "").strip()

    print(f"Running Question {question_id}...")

    # Same Direct Prompt we already tested
    messages = [
        {
            "role": "user",
            "content": f"""Solve the following math word problem.

Do not provide an explanation or show your working.
Return only the final numerical answer in this exact format:

Final Answer: <number>

Problem:
{question}"""
        }
    ]

    # Format prompt for Qwen
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Remove the original prompt
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    model_response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # Extract the number after "Final Answer:"
    match = re.search(
        r"Final Answer:\s*\$?(-?[\d,]+(?:\.\d+)?)",
        model_response,
        re.IGNORECASE
    )

    if match:
        model_answer = match.group(1).replace(",", "")
    else:
        model_answer = None

    # Compare model answer with GSM8K answer
    is_correct = model_answer == correct_answer

    # Save result
    results.append({
        "id": question_id,
        "question": question,
        "correct_answer": correct_answer,
        "model_response": model_response,
        "model_answer": model_answer,
        "correct": is_correct
    })

    # Print result
    print("Model Answer:", model_answer)
    print("Correct Answer:", correct_answer)
    print("Result:", "CORRECT" if is_correct else "WRONG")
    print("-----------------------------------")


# Turn results into a table
results_df = pd.DataFrame(results)

# Calculate accuracy
accuracy = results_df["correct"].mean() * 100

print("\n==============================")
print("DIRECT PROMPTING TEST COMPLETE")
print("==============================")
print(f"Correct: {results_df['correct'].sum()} / {len(results_df)}")
print(f"Accuracy: {accuracy:.2f}%")

Running Question 1...
Model Answer: 82
Correct Answer: 109
Result: WRONG
-----------------------------------
Running Question 2...
Model Answer: 123
Correct Answer: 89
Result: WRONG
-----------------------------------
Running Question 3...
Model Answer: 14
Correct Answer: 13
Result: WRONG
-----------------------------------
Running Question 4...
Model Answer: 16
Correct Answer: 5
Result: WRONG
-----------------------------------
Running Question 5...
Model Answer: 15
Correct Answer: 25
Result: WRONG
-----------------------------------

DIRECT PROMPTING TEST COMPLETE
Correct: 0 / 5
Accuracy: 0.00%


In [ ]:
# ============================================================
# OFFICIAL DIRECT PROMPTING EXPERIMENT - ALL 200 GSM8K QUESTIONS
# Qwen2.5-3B-Instruct
# Checkpoint saving + resume support + final CSV download
# ============================================================

import os
import re
import torch
import pandas as pd
from google.colab import files

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

RESULTS_FILE = "/content/direct_prompting_results.csv"
TOTAL_QUESTIONS = 200

# ------------------------------------------------------------
# FUNCTION: NORMALIZE ANSWERS
# This makes comparisons cleaner.
# Example: "1,200" becomes "1200"
# ------------------------------------------------------------

def normalize_answer(answer):
    if answer is None:
        return None

    answer = str(answer).strip()
    answer = answer.replace(",", "")
    answer = answer.replace("$", "")

    # Make 13.0 equivalent to 13
    try:
        number = float(answer)

        if number.is_integer():
            return str(int(number))

        return str(number)

    except:
        return answer


# ------------------------------------------------------------
# FUNCTION: EXTRACT "Final Answer: <number>"
# ------------------------------------------------------------

def extract_final_answer(response):

    match = re.search(
        r"Final Answer:\s*\$?(-?[\d,]+(?:\.\d+)?)",
        response,
        re.IGNORECASE
    )

    if match:
        return normalize_answer(match.group(1))

    return None


# ------------------------------------------------------------
# RESUME SUPPORT
# ------------------------------------------------------------

if os.path.exists(RESULTS_FILE):

    results_df = pd.read_csv(RESULTS_FILE)

    # Convert previous results back into dictionaries
    results = results_df.to_dict("records")

    completed_ids = set(results_df["id"].astype(int).tolist())

    print("Existing checkpoint found.")
    print(f"Already completed: {len(completed_ids)} questions.")
    print("Resuming experiment...\n")

else:

    results = []
    completed_ids = set()

    print("No existing checkpoint found.")
    print("Starting Direct Prompting experiment from Question 1...\n")


# ------------------------------------------------------------
# RUN ALL 200 QUESTIONS
# ------------------------------------------------------------

for index, row in df.head(TOTAL_QUESTIONS).iterrows():

    question_id = int(row["id"])

    # Skip questions already completed
    if question_id in completed_ids:
        continue

    question = row["question"]

    correct_answer = normalize_answer(
        row["correct_answer"]
    )

    print(f"Running Question {question_id}/{TOTAL_QUESTIONS}...")


    # --------------------------------------------------------
    # FIXED DIRECT PROMPT
    # DO NOT CHANGE DURING THE EXPERIMENT
    # --------------------------------------------------------

    messages = [
        {
            "role": "user",
            "content": f"""Solve the following math word problem.

Do not provide an explanation or show your working.
Return only the final numerical answer in this exact format:

Final Answer: <number>

Problem:
{question}"""
        }
    ]


    # --------------------------------------------------------
    # FORMAT FOR QWEN
    # --------------------------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)


    # --------------------------------------------------------
    # GENERATE ANSWER
    # Deterministic generation for Direct Prompting
    # --------------------------------------------------------

    try:

        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )


        # Remove the original prompt
        generated_tokens = outputs[0][
            inputs["input_ids"].shape[1]:
        ]


        # Decode model response
        model_response = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()


        # Extract final numerical answer
        model_answer = extract_final_answer(
            model_response
        )


        # Compare against GSM8K
        is_correct = (
            model_answer == correct_answer
        )


        # ----------------------------------------------------
        # SAVE RESULT
        # ----------------------------------------------------

        result = {
            "id": question_id,
            "question": question,
            "correct_answer": correct_answer,
            "model_response": model_response,
            "model_answer": model_answer,
            "correct": is_correct
        }

        results.append(result)


        # ----------------------------------------------------
        # CHECKPOINT AFTER EVERY QUESTION
        # ----------------------------------------------------

        results_df = pd.DataFrame(results)

        results_df.to_csv(
            RESULTS_FILE,
            index=False
        )


        # ----------------------------------------------------
        # PRINT PROGRESS
        # ----------------------------------------------------

        print("Model Answer:", model_answer)
        print("Correct Answer:", correct_answer)

        print(
            "Result:",
            "CORRECT" if is_correct else "WRONG"
        )

        current_correct = results_df["correct"].sum()

        current_accuracy = (
            current_correct / len(results_df)
        ) * 100

        print(
            f"Running Accuracy: "
            f"{current_correct}/{len(results_df)} "
            f"({current_accuracy:.2f}%)"
        )

        print("-----------------------------------")


        # ----------------------------------------------------
        # BACKUP DOWNLOAD EVERY 20 QUESTIONS
        # ----------------------------------------------------

        if len(results_df) % 20 == 0:

            print(
                f"\nCheckpoint reached: "
                f"{len(results_df)} questions."
            )

            print("Downloading backup CSV...\n")

            files.download(RESULTS_FILE)


    except Exception as e:

        print(f"\nERROR on Question {question_id}")
        print(e)

        print(
            "\nYour completed results have already "
            "been saved."
        )

        print(
            "Run this cell again to resume."
        )

        break


# ------------------------------------------------------------
# FINAL RESULTS
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

if len(results_df) > 0:

    total_completed = len(results_df)

    total_correct = int(
        results_df["correct"].sum()
    )

    accuracy = (
        total_correct / total_completed
    ) * 100


    print("\n======================================")
    print("DIRECT PROMPTING RESULTS")
    print("======================================")

    print(
        f"Questions completed: "
        f"{total_completed}/{TOTAL_QUESTIONS}"
    )

    print(
        f"Correct answers: "
        f"{total_correct}/{total_completed}"
    )

    print(
        f"Accuracy: {accuracy:.2f}%"
    )

    print(
        f"Answer extraction failures: "
        f"{results_df['model_answer'].isna().sum()}"
    )


    # --------------------------------------------------------
    # DOWNLOAD FINAL FILE IF ALL 200 FINISHED
    # --------------------------------------------------------

    if total_completed == TOTAL_QUESTIONS:

        print("\n🎉 ALL 200 QUESTIONS COMPLETE!")

        print(
            "Downloading final results CSV..."
        )

        files.download(RESULTS_FILE)

    else:

        print(
            "\nExperiment is not finished yet."
        )

        print(
            "Run this cell again to continue "
            "from the checkpoint."
        )

No existing checkpoint found.
Starting Direct Prompting experiment from Question 1...

Running Question 1/200...
Model Answer: 82
Correct Answer: 109
Result: WRONG
Running Accuracy: 0/1 (0.00%)
-----------------------------------
Running Question 2/200...
Model Answer: 123
Correct Answer: 89
Result: WRONG
Running Accuracy: 0/2 (0.00%)
-----------------------------------
Running Question 3/200...
Model Answer: 14
Correct Answer: 13
Result: WRONG
Running Accuracy: 0/3 (0.00%)
-----------------------------------
Running Question 4/200...
Model Answer: 16
Correct Answer: 5
Result: WRONG
Running Accuracy: 0/4 (0.00%)
-----------------------------------
Running Question 5/200...
Model Answer: 15
Correct Answer: 25
Result: WRONG
Running Accuracy: 0/5 (0.00%)
-----------------------------------
Running Question 6/200...
Model Answer: 475
Correct Answer: 452
Result: WRONG
Running Accuracy: 0/6 (0.00%)
-----------------------------------
Running Question 7/200...
Model Answer: 57
Correct Answer:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Running Question 21/200...
Model Answer: 75
Correct Answer: 45
Result: WRONG
Running Accuracy: 0/21 (0.00%)
-----------------------------------
Running Question 22/200...
Model Answer: 10
Correct Answer: 8
Result: WRONG
Running Accuracy: 0/22 (0.00%)
-----------------------------------
Running Question 23/200...
Model Answer: 60
Correct Answer: 36
Result: WRONG
Running Accuracy: 0/23 (0.00%)
-----------------------------------
Running Question 24/200...
Model Answer: 2
Correct Answer: 1
Result: WRONG
Running Accuracy: 0/24 (0.00%)
-----------------------------------
Running Question 25/200...
Model Answer: 30
Correct Answer: 18
Result: WRONG
Running Accuracy: 0/25 (0.00%)
-----------------------------------
Running Question 26/200...
Model Answer: 34
Correct Answer: 33
Result: WRONG
Running Accuracy: 0/26 (0.00%)
-----------------------------------
Running Question 27/200...
Model Answer: 1440
Correct Answer: 1248
Result: WRONG
Running Accuracy: 0/27 (0.00%)
---------------------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (23 outputs left out)

<IPython.core.display.Javascript object>


DIRECT PROMPTING RESULTS
Questions completed: 200/200
Correct answers: 24/200
Accuracy: 12.00%
Answer extraction failures: 0

🎉 ALL 200 QUESTIONS COMPLETE!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==========================================
# ZERO-SHOT CHAIN-OF-THOUGHT
# TEST QUESTION 1
# ==========================================

import torch

# Get Question 1
question = df.iloc[0]["question"]
correct_answer = df.iloc[0]["correct_answer"]

# Zero-shot Chain-of-Thought prompt
messages = [
    {
        "role": "user",
        "content": f"""Solve the following math word problem.

Work through the problem step by step.

After your reasoning, give the final numerical answer in this exact format:

Final Answer: <number>

Problem:
{question}"""
    }
]

# Format correctly for Qwen
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Generate response
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Remove original prompt
generated_tokens = outputs[0][
    inputs["input_ids"].shape[1]:
]

# Decode response
model_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

# Display results
print("QUESTION:")
print(question)

print("\nMODEL RESPONSE:")
print(model_response)

print("\nCORRECT GSM8K ANSWER:")
print(correct_answer)

QUESTION:
Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now.

MODEL RESPONSE:
To solve the problem, we will follow these steps:

1. **Understand the given information:**
   - The ratio of Darrell's age to Allen's age is \(7:11\).
   - The sum of their current ages is \(162\).

2. **Set up the variables:**
   Let \(D\) be Darrell's current age.
   Let \(A\) be Allen's current age.

3. **Express the ratio in terms of a common variable:**
   Since the ratio of their ages is \(7:11\), we can write:
   \[
   D = 7k \quad \text{and} \quad A = 11k
   \]
   where \(k\) is a common multiplier.

4. **Use the total age equation:**
   According to the problem, the sum of their ages is \(162\):
   \[
   D + A = 162
   \]

5. **Substitute the expressions for \(D\) and \(A\) into the total age equation:**
   \[
   7k + 11k = 162
   \]

6. **Combine like terms:**
   \[
   18k = 162
   \]

7. **Solve for \(k\):**
   \[
   k = \frac

In [ ]:
# ==========================================
# ZERO-SHOT CoT - TEST QUESTIONS 1 TO 5
# ==========================================

import torch
import re
import pandas as pd


# ------------------------------------------
# FUNCTION: Normalize answers
# ------------------------------------------

def normalize_answer(answer):

    if answer is None:
        return None

    answer = str(answer).strip()
    answer = answer.replace(",", "")
    answer = answer.replace("$", "")

    try:
        number = float(answer)

        if number.is_integer():
            return str(int(number))

        return str(number)

    except:
        return answer


# ------------------------------------------
# FUNCTION: Extract final answer
# ------------------------------------------

def extract_cot_answer(response):

    # 1. Look for boxed answer
    # Example: \boxed{109}

    boxed_matches = re.findall(
        r"\\boxed\{\s*\$?(-?[\d,]+(?:\.\d+)?)\s*\}",
        response
    )

    if boxed_matches:
        return normalize_answer(boxed_matches[-1])


    # 2. Look for explicit Final Answer
    # Example: Final Answer: 109

    final_matches = re.findall(
        r"Final Answer:\s*\$?(-?[\d,]+(?:\.\d+)?)",
        response,
        re.IGNORECASE
    )

    if final_matches:
        return normalize_answer(final_matches[-1])


    # 3. FALLBACK
    # Look for the LAST equation ending in a number
    # Example: 13 + 13 = 26

    equation_matches = re.findall(
        r"=\s*\$?(-?[\d,]+(?:\.\d+)?)",
        response
    )

    if equation_matches:
        return normalize_answer(equation_matches[-1])


    # If nothing can be extracted
    return None


# ------------------------------------------
# RUN QUESTIONS 1 TO 5
# ------------------------------------------

results_cot = []

for index, row in df.head(5).iterrows():

    question_id = row["id"]
    question = row["question"]

    correct_answer = normalize_answer(
        row["correct_answer"]
    )

    print(f"Running Question {question_id}...")


    # --------------------------------------
    # ZERO-SHOT CHAIN-OF-THOUGHT PROMPT
    # --------------------------------------

    messages = [
        {
            "role": "user",
            "content": f"""Solve the following math word problem.

Work through the problem step by step.

After your reasoning, give the final numerical answer in this exact format:

Final Answer: <number>

Problem:
{question}"""
        }
    ]


    # --------------------------------------
    # FORMAT PROMPT FOR QWEN
    # --------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)


    # --------------------------------------
    # GENERATE RESPONSE
    # --------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )


    # Remove original prompt
    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]


    # Decode model response
    model_response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


    # --------------------------------------
    # EXTRACT MODEL ANSWER
    # --------------------------------------

    model_answer = extract_cot_answer(
        model_response
    )


    # --------------------------------------
    # COMPARE WITH GSM8K ANSWER
    # --------------------------------------

    is_correct = (
        model_answer == correct_answer
    )


    # --------------------------------------
    # SAVE RESULT
    # --------------------------------------

    results_cot.append({
        "id": question_id,
        "question": question,
        "correct_answer": correct_answer,
        "model_response": model_response,
        "model_answer": model_answer,
        "correct": is_correct
    })


    # --------------------------------------
    # SHOW RESULT
    # --------------------------------------

    print("Model Answer:", model_answer)
    print("Correct Answer:", correct_answer)

    print(
        "Result:",
        "CORRECT" if is_correct else "WRONG"
    )

    print("-----------------------------------")


# ==========================================
# CREATE RESULTS TABLE
# ==========================================

cot_test_df = pd.DataFrame(results_cot)


# Calculate accuracy
correct_count = int(
    cot_test_df["correct"].sum()
)

accuracy = (
    correct_count / len(cot_test_df)
) * 100


# Count extraction failures
extraction_failures = (
    cot_test_df["model_answer"]
    .isna()
    .sum()
)


# ==========================================
# FINAL TEST RESULTS
# ==========================================

print("\n================================")
print("ZERO-SHOT CoT TEST COMPLETE")
print("================================")

print(
    f"Correct: {correct_count} / "
    f"{len(cot_test_df)}"
)

print(
    f"Accuracy: {accuracy:.2f}%"
)

print(
    f"Extraction failures: "
    f"{extraction_failures}"
)

Running Question 1...
Model Answer: 109
Correct Answer: 109
Result: CORRECT
-----------------------------------
Running Question 2...
Model Answer: 26
Correct Answer: 89
Result: WRONG
-----------------------------------
Running Question 3...
Model Answer: 13
Correct Answer: 13
Result: CORRECT
-----------------------------------
Running Question 4...
Model Answer: 5
Correct Answer: 5
Result: CORRECT
-----------------------------------
Running Question 5...
Model Answer: 0
Correct Answer: 25
Result: WRONG
-----------------------------------

ZERO-SHOT CoT TEST COMPLETE
Correct: 3 / 5
Accuracy: 60.00%
Extraction failures: 0


In [ ]:
# ============================================================
# OFFICIAL ZERO-SHOT CHAIN-OF-THOUGHT EXPERIMENT
# ALL 200 GSM8K QUESTIONS
#
# Model: Qwen2.5-3B-Instruct
# Checkpoint saving + resume support + automatic backups
# ============================================================

import os
import re
import torch
import pandas as pd
from google.colab import files


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

RESULTS_FILE = "/content/zero_shot_cot_results.csv"
TOTAL_QUESTIONS = 200


# ------------------------------------------------------------
# NORMALIZE ANSWERS
# ------------------------------------------------------------

def normalize_answer(answer):

    if answer is None:
        return None

    answer = str(answer).strip()
    answer = answer.replace(",", "")
    answer = answer.replace("$", "")

    try:
        number = float(answer)

        if number.is_integer():
            return str(int(number))

        return str(number)

    except:
        return answer


# ------------------------------------------------------------
# EXTRACT FINAL ANSWER
# ------------------------------------------------------------

def extract_cot_answer(response):

    # 1. Look for boxed answer
    # Example: \boxed{109}

    boxed_matches = re.findall(
        r"\\boxed\{\s*\$?(-?[\d,]+(?:\.\d+)?)\s*\}",
        response
    )

    if boxed_matches:
        return normalize_answer(boxed_matches[-1])


    # 2. Look for explicit Final Answer
    # Example: Final Answer: 109

    final_matches = re.findall(
        r"Final Answer:\s*\$?(-?[\d,]+(?:\.\d+)?)",
        response,
        re.IGNORECASE
    )

    if final_matches:
        return normalize_answer(final_matches[-1])


    # 3. Fallback:
    # Find the LAST equation ending in a number
    # Example: 13 + 13 = 26

    equation_matches = re.findall(
        r"=\s*\$?(-?[\d,]+(?:\.\d+)?)",
        response
    )

    if equation_matches:
        return normalize_answer(equation_matches[-1])


    return None


# ------------------------------------------------------------
# CHECK FOR EXISTING CHECKPOINT
# ------------------------------------------------------------

if os.path.exists(RESULTS_FILE):

    results_df = pd.read_csv(RESULTS_FILE)

    results = results_df.to_dict("records")

    completed_ids = set(
        results_df["id"].astype(int).tolist()
    )

    print("Existing Zero-shot CoT checkpoint found.")
    print(
        f"Already completed: "
        f"{len(completed_ids)} questions."
    )

    print("Resuming experiment...\n")


else:

    results = []
    completed_ids = set()

    print("No existing Zero-shot CoT checkpoint found.")
    print("Starting from Question 1...\n")


# ------------------------------------------------------------
# RUN ALL 200 QUESTIONS
# ------------------------------------------------------------

for index, row in df.head(TOTAL_QUESTIONS).iterrows():

    question_id = int(row["id"])


    # --------------------------------------------------------
    # SKIP ALREADY COMPLETED QUESTIONS
    # --------------------------------------------------------

    if question_id in completed_ids:
        continue


    question = row["question"]

    correct_answer = normalize_answer(
        row["correct_answer"]
    )


    print(
        f"Running Question "
        f"{question_id}/{TOTAL_QUESTIONS}..."
    )


    # --------------------------------------------------------
    # FIXED ZERO-SHOT CHAIN-OF-THOUGHT PROMPT
    #
    # DO NOT CHANGE THIS DURING THE EXPERIMENT
    # --------------------------------------------------------

    messages = [
        {
            "role": "user",
            "content": f"""Solve the following math word problem.

Work through the problem step by step.

After your reasoning, give the final numerical answer in this exact format:

Final Answer: <number>

Problem:
{question}"""
        }
    ]


    # --------------------------------------------------------
    # FORMAT PROMPT FOR QWEN
    # --------------------------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)


    # --------------------------------------------------------
    # GENERATE RESPONSE
    #
    # Same deterministic setting tested earlier.
    # --------------------------------------------------------

    try:

        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )


        # ----------------------------------------------------
        # REMOVE ORIGINAL PROMPT FROM OUTPUT
        # ----------------------------------------------------

        generated_tokens = outputs[0][
            inputs["input_ids"].shape[1]:
        ]


        # ----------------------------------------------------
        # DECODE MODEL RESPONSE
        # ----------------------------------------------------

        model_response = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()


        # ----------------------------------------------------
        # EXTRACT FINAL NUMERICAL ANSWER
        # ----------------------------------------------------

        model_answer = extract_cot_answer(
            model_response
        )


        # ----------------------------------------------------
        # COMPARE WITH GSM8K
        # ----------------------------------------------------

        is_correct = (
            model_answer == correct_answer
        )


        # ----------------------------------------------------
        # SAVE RESULT
        # ----------------------------------------------------

        result = {

            "id": question_id,

            "question": question,

            "correct_answer": correct_answer,

            "model_response": model_response,

            "model_answer": model_answer,

            "correct": is_correct
        }


        results.append(result)


        # ----------------------------------------------------
        # SAVE CHECKPOINT AFTER EVERY QUESTION
        # ----------------------------------------------------

        results_df = pd.DataFrame(results)


        results_df.to_csv(
            RESULTS_FILE,
            index=False
        )


        # ----------------------------------------------------
        # SHOW CURRENT RESULT
        # ----------------------------------------------------

        print(
            "Model Answer:",
            model_answer
        )

        print(
            "Correct Answer:",
            correct_answer
        )

        print(
            "Result:",
            "CORRECT" if is_correct else "WRONG"
        )


        # ----------------------------------------------------
        # RUNNING ACCURACY
        # ----------------------------------------------------

        current_correct = int(
            results_df["correct"].sum()
        )


        current_accuracy = (
            current_correct /
            len(results_df)
        ) * 100


        print(
            f"Running Accuracy: "
            f"{current_correct}/"
            f"{len(results_df)} "
            f"({current_accuracy:.2f}%)"
        )


        # ----------------------------------------------------
        # CURRENT EXTRACTION FAILURES
        # ----------------------------------------------------

        current_failures = (
            results_df["model_answer"]
            .isna()
            .sum()
        )


        print(
            f"Extraction Failures: "
            f"{current_failures}"
        )


        print(
            "-----------------------------------"
        )


        # ----------------------------------------------------
        # DOWNLOAD BACKUP EVERY 20 QUESTIONS
        # ----------------------------------------------------

        if len(results_df) % 20 == 0:

            print(
                f"\nCheckpoint reached: "
                f"{len(results_df)} questions."
            )

            print(
                "Downloading backup CSV...\n"
            )

            files.download(
                RESULTS_FILE
            )


    # --------------------------------------------------------
    # IF SOMETHING GOES WRONG
    # --------------------------------------------------------

    except Exception as e:

        print(
            f"\nERROR on Question "
            f"{question_id}"
        )

        print(e)


        print(
            "\nCompleted results have already "
            "been saved."
        )

        print(
            "Run this cell again to resume "
            "from the checkpoint."
        )

        break


# ============================================================
# FINAL RESULTS
# ============================================================

results_df = pd.DataFrame(results)


if len(results_df) > 0:

    total_completed = len(
        results_df
    )


    total_correct = int(
        results_df["correct"].sum()
    )


    accuracy = (
        total_correct /
        total_completed
    ) * 100


    extraction_failures = (
        results_df["model_answer"]
        .isna()
        .sum()
    )


    print(
        "\n======================================"
    )

    print(
        "ZERO-SHOT CHAIN-OF-THOUGHT RESULTS"
    )

    print(
        "======================================"
    )


    print(
        f"Questions completed: "
        f"{total_completed}/"
        f"{TOTAL_QUESTIONS}"
    )


    print(
        f"Correct answers: "
        f"{total_correct}/"
        f"{total_completed}"
    )


    print(
        f"Accuracy: "
        f"{accuracy:.2f}%"
    )


    print(
        f"Answer extraction failures: "
        f"{extraction_failures}"
    )


    # --------------------------------------------------------
    # DOWNLOAD FINAL RESULTS
    # --------------------------------------------------------

    if total_completed == TOTAL_QUESTIONS:

        print(
            "\n🎉 ALL 200 ZERO-SHOT CoT "
            "QUESTIONS COMPLETE!"
        )

        print(
            "Downloading final results CSV..."
        )


        files.download(
            RESULTS_FILE
        )


    else:

        print(
            "\nExperiment is not finished yet."
        )

        print(
            "Run this cell again to continue "
            "from the checkpoint."
        )

No existing Zero-shot CoT checkpoint found.
Starting from Question 1...

Running Question 1/200...
Model Answer: 109
Correct Answer: 109
Result: CORRECT
Running Accuracy: 1/1 (100.00%)
Extraction Failures: 0
-----------------------------------
Running Question 2/200...
Model Answer: 26
Correct Answer: 89
Result: WRONG
Running Accuracy: 1/2 (50.00%)
Extraction Failures: 0
-----------------------------------
Running Question 3/200...
Model Answer: 13
Correct Answer: 13
Result: CORRECT
Running Accuracy: 2/3 (66.67%)
Extraction Failures: 0
-----------------------------------
Running Question 4/200...
Model Answer: 5
Correct Answer: 5
Result: CORRECT
Running Accuracy: 3/4 (75.00%)
Extraction Failures: 0
-----------------------------------
Running Question 5/200...
Model Answer: 0
Correct Answer: 25
Result: WRONG
Running Accuracy: 3/5 (60.00%)
Extraction Failures: 0
-----------------------------------
Running Question 6/200...
Model Answer: 462
Correct Answer: 452
Result: WRONG
Running Accur

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Running Question 21/200...
Model Answer: 45
Correct Answer: 45
Result: CORRECT
Running Accuracy: 16/21 (76.19%)
Extraction Failures: 0
-----------------------------------
Running Question 22/200...
Model Answer: 8
Correct Answer: 8
Result: CORRECT
Running Accuracy: 17/22 (77.27%)
Extraction Failures: 0
-----------------------------------
Running Question 23/200...
Model Answer: 36
Correct Answer: 36
Result: CORRECT
Running Accuracy: 18/23 (78.26%)
Extraction Failures: 0
-----------------------------------
Running Question 24/200...
Model Answer: 1
Correct Answer: 1
Result: CORRECT
Running Accuracy: 19/24 (79.17%)
Extraction Failures: 0
-----------------------------------
Running Question 25/200...
Model Answer: 10
Correct Answer: 18
Result: WRONG
Running Accuracy: 19/25 (76.00%)
Extraction Failures: 0
-----------------------------------
Running Question 26/200...
Model Answer: 20
Correct Answer: 33
Result: WRONG
Running Accuracy: 19/26 (73.08%)
Extraction Failures: 0
------------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (23 outputs left out)

<IPython.core.display.Javascript object>


ZERO-SHOT CHAIN-OF-THOUGHT RESULTS
Questions completed: 200/200
Correct answers: 167/200
Accuracy: 83.50%
Answer extraction failures: 2

🎉 ALL 200 ZERO-SHOT CoT QUESTIONS COMPLETE!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ==========================================
# SHOW ZERO-SHOT CoT EXTRACTION FAILURES
# ==========================================

failures = results_df[
    results_df["model_answer"].isna()
]

print("Number of extraction failures:", len(failures))

for _, row in failures.iterrows():

    print("\n" + "=" * 70)

    print("QUESTION ID:", row["id"])

    print("\nQUESTION:")
    print(row["question"])

    print("\nRAW MODEL RESPONSE:")
    print(row["model_response"])

    print("\nCORRECT GSM8K ANSWER:")
    print(row["correct_answer"])

    print("=" * 70)

Number of extraction failures: 2

QUESTION ID: 28

QUESTION:
Jerry is rolling a six-sided die. How much more likely is it (expressed as a percentage) that he rolls a number greater than 3 than that he rolls two even numbers in a row?

RAW MODEL RESPONSE:
To solve this problem, we need to calculate the probabilities of two events and then compare them to find out how much more likely one event is compared to the other.

### Step 1: Calculate the probability of rolling a number greater than 3

A six-sided die has the numbers 1, 2, 3, 4, 5, and 6. The numbers greater than 3 are 4, 5, and 6.

- Total possible outcomes when rolling a die: 6
- Favorable outcomes for rolling a number greater than 3: 4, 5, 6 (3 outcomes)

The probability \( P(\text{greater than 3}) \) is given by:
\[
P(\text{greater than 3}) = \frac{\text{Number of favorable outcomes}}{\text{Total number of outcomes}} = \frac{3}{6} = \frac{1}{2}
\]

### Step 2: Calculate the probability of rolling two even numbers in a row

Fi

In [ ]:
# ============================================================
# ZERO-SHOT CoT - FINAL POST-PROCESSING AUDIT
# NO MODEL CALLS - uses the 200 responses already generated
# ============================================================

import re
import pandas as pd
from google.colab import files


# ------------------------------------------------------------
# NORMALIZE ANSWERS
# ------------------------------------------------------------

def normalize_final_answer(answer):

    if answer is None:
        return None

    answer = str(answer).strip()
    answer = answer.replace(",", "")
    answer = answer.replace("$", "")

    try:
        number = float(answer)

        if number.is_integer():
            return str(int(number))

        return str(number)

    except:
        return answer


# ------------------------------------------------------------
# IMPROVED ANSWER EXTRACTOR
# ------------------------------------------------------------

def final_cot_extractor(response):

    if pd.isna(response):
        return None

    response = str(response)


    # 1. \boxed{109}
    boxed = re.findall(
        r"\\boxed\{\s*\\?\$?\s*(-?[\d,]+(?:\.\d+)?)\s*\}",
        response
    )

    if boxed:
        return normalize_final_answer(boxed[-1])


    # 2. Final Answer: 109
    # Also handles:
    # Final Answer: $9.00
    # Final Answer: \( $9.00 \)

    final_answer = re.findall(
        r"Final\s+Answer\s*:\s*"
        r"(?:\\\(\s*)?"
        r"\\?\$?\s*"
        r"(-?[\d,]+(?:\.\d+)?)",
        response,
        re.IGNORECASE
    )

    if final_answer:
        return normalize_final_answer(final_answer[-1])


    # 3. Phrases such as:
    # "the final numerical answer is 25"
    # "the answer is 25"

    answer_phrase = re.findall(
        r"(?:final numerical answer|final answer|answer)\s+is\s*"
        r"(?:\\\(\s*)?"
        r"\\?\$?\s*"
        r"(-?[\d,]+(?:\.\d+)?)",
        response,
        re.IGNORECASE
    )

    if answer_phrase:
        return normalize_final_answer(answer_phrase[-1])


    # 4. Fallback to final completed equation
    # Example: 5 + 3 + 1 = 9

    equations = re.findall(
        r"=\s*"
        r"(?:\\\(\s*)?"
        r"\\?\$?\s*"
        r"(-?[\d,]+(?:\.\d+)?)"
        r"(?![\d.])",
        response
    )

    if equations:
        return normalize_final_answer(equations[-1])


    # Nothing reliable was found
    return None


# ============================================================
# LOAD THE COMPLETED ZERO-SHOT RESULTS
# ============================================================

audit_df = pd.read_csv(
    "/content/zero_shot_cot_results.csv"
)


# Keep the ORIGINAL extracted answer for comparison
audit_df["original_model_answer"] = audit_df["model_answer"]


# ------------------------------------------------------------
# RE-EXTRACT ALL 200 RESPONSES
# ------------------------------------------------------------

audit_df["model_answer"] = audit_df["model_response"].apply(
    final_cot_extractor
)


# Normalize GSM8K answers
audit_df["correct_answer"] = audit_df[
    "correct_answer"
].apply(normalize_final_answer)


# Recalculate correctness
audit_df["correct"] = (
    audit_df["model_answer"] ==
    audit_df["correct_answer"]
)


# ============================================================
# FIND RESPONSES WHERE EXTRACTION CHANGED
# ============================================================

changed = audit_df[
    audit_df["original_model_answer"].fillna("NONE").astype(str)
    !=
    audit_df["model_answer"].fillna("NONE").astype(str)
]


print("======================================")
print("POST-PROCESSING AUDIT")
print("======================================")

print(
    "Responses whose extracted answer changed:",
    len(changed)
)

if len(changed) > 0:

    print("\nChanged responses:")

    for _, row in changed.iterrows():

        print("\n-----------------------------------")

        print("Question ID:", row["id"])

        print(
            "Original extraction:",
            row["original_model_answer"]
        )

        print(
            "New extraction:",
            row["model_answer"]
        )

        print(
            "Correct answer:",
            row["correct_answer"]
        )

        print(
            "Result:",
            "CORRECT" if row["correct"] else "WRONG"
        )


# ============================================================
# FINAL ACCURACY
# ============================================================

total = len(audit_df)

correct = int(
    audit_df["correct"].sum()
)

accuracy = (
    correct / total
) * 100

failures = int(
    audit_df["model_answer"].isna().sum()
)


print("\n======================================")
print("FINAL ZERO-SHOT CoT RESULTS")
print("======================================")

print(f"Questions: {total}")
print(f"Correct: {correct}/{total}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Remaining extraction failures: {failures}")


# ------------------------------------------------------------
# SHOW ANY REMAINING FAILURES
# ------------------------------------------------------------

remaining_failures = audit_df[
    audit_df["model_answer"].isna()
]

if len(remaining_failures) > 0:

    print("\nRemaining failure IDs:")

    print(
        remaining_failures["id"].tolist()
    )


# ============================================================
# SAVE CLEAN FINAL RESULTS
# ============================================================

FINAL_FILE = (
    "/content/zero_shot_cot_FINAL_200.csv"
)

audit_df.to_csv(
    FINAL_FILE,
    index=False
)

print(
    "\nSaved:",
    FINAL_FILE
)

files.download(
    FINAL_FILE
)

POST-PROCESSING AUDIT
Responses whose extracted answer changed: 197

Changed responses:

-----------------------------------
Question ID: 1
Original extraction: 109.0
New extraction: 109
Correct answer: 109
Result: CORRECT

-----------------------------------
Question ID: 2
Original extraction: 26.0
New extraction: 26
Correct answer: 89
Result: WRONG

-----------------------------------
Question ID: 3
Original extraction: 13.0
New extraction: 13
Correct answer: 13
Result: CORRECT

-----------------------------------
Question ID: 4
Original extraction: 5.0
New extraction: 5
Correct answer: 5
Result: CORRECT

-----------------------------------
Question ID: 5
Original extraction: 0.0
New extraction: 0
Correct answer: 25
Result: WRONG

-----------------------------------
Question ID: 6
Original extraction: 462.0
New extraction: 462
Correct answer: 452
Result: WRONG

-----------------------------------
Question ID: 7
Original extraction: 43.0
New extraction: 43
Correct answer: 43
Result: C

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Kekweditse section: Few shot CoT cell setup

In [ ]:
# ============================================================
# FEW-SHOT CHAIN-OF-THOUGHT — SETUP
# ============================================================

import os
import re
import math
import json
import pandas as pd
import torch

from datasets import load_dataset


# Confirm we are using the correct model and exact 200 questions
assert MODEL_NAME == "Qwen/Qwen2.5-3B-Instruct"
assert len(df) == 200

print("Model:", MODEL_NAME)
print("Evaluation questions:", len(df))


# ------------------------------------------------------------
# Load GSM8K TRAINING split for few-shot examples
# ------------------------------------------------------------

gsm8k_train = load_dataset(
    "openai/gsm8k",
    "main",
    split="train"
)

# SAME four examples will be used for every question
FEWSHOT_TRAIN_INDICES = [0, 1, 2, 3]

FEWSHOT_EXAMPLES = [
    gsm8k_train[i]
    for i in FEWSHOT_TRAIN_INDICES
]


# ------------------------------------------------------------
# Make sure none of the examples are in our 200 test questions
# ------------------------------------------------------------

test_questions = set(
    df["question"].astype(str).str.strip()
)

for idx, example in zip(
    FEWSHOT_TRAIN_INDICES,
    FEWSHOT_EXAMPLES
):
    assert example["question"].strip() not in test_questions, (
        f"DATA LEAKAGE detected at training index {idx}"
    )

print("Leakage check PASSED.")
print("Fixed training examples:", FEWSHOT_TRAIN_INDICES)

Model: Qwen/Qwen2.5-3B-Instruct
Evaluation questions: 200
Leakage check PASSED.
Fixed training examples: [0, 1, 2, 3]


Answer extraction

In [ ]:
# ============================================================
# ANSWER NORMALIZATION + EXTRACTION
# ============================================================

def normalize_answer(answer):

    if answer is None:
        return None

    if isinstance(answer, float) and math.isnan(answer):
        return None

    answer = str(answer).strip()

    answer = answer.replace(",", "")
    answer = answer.replace("$", "")

    try:
        number = float(answer)

        if number.is_integer():
            return str(int(number))

        return str(number)

    except:
        return answer


def extract_final_answer(response):

    if response is None:
        return None

    if pd.isna(response):
        return None

    response = str(response)

    # 1. Preferred format: Final Answer: <number>
    final_answer = re.findall(
        r"Final\s+Answer\s*:\s*"
        r"(?:\\\(\s*)?"
        r"\\?\$?\s*"
        r"(-?[\d,]+(?:\.\d+)?)",
        response,
        re.IGNORECASE
    )

    if final_answer:
        return normalize_answer(final_answer[-1])

    # 2. Accept boxed numerical answer
    boxed = re.findall(
        r"\\boxed\{\s*\\?\$?\s*(-?[\d,]+(?:\.\d+)?)\s*\}",
        response
    )

    if boxed:
        return normalize_answer(boxed[-1])

    # If no explicit final answer is found,
    # record it as an extraction failure.
    return None


print("Strict final-answer extractor ready.")

Strict final-answer extractor ready.


Few shot prompt

In [ ]:
# ============================================================
# BUILD FIXED FEW-SHOT CoT PROMPT
# ============================================================

def prepare_training_answer(answer_text):

    answer_text = str(answer_text).strip()

    # GSM8K answers finish with:
    # #### final_answer

    match = re.search(
        r"####\s*(-?[\d,]+(?:\.\d+)?)\s*$",
        answer_text
    )

    if match is None:
        raise ValueError(
            "Could not extract training example answer."
        )

    final_answer = normalize_answer(
        match.group(1)
    )

    reasoning = re.sub(
        r"\n?####\s*.+?\s*$",
        "",
        answer_text
    ).strip()

    return (
        reasoning
        + "\nFinal Answer: "
        + final_answer
    )


def build_fewshot_messages(question):

    messages = []

    # Add the SAME four worked examples
    # for every evaluation question.

    for example in FEWSHOT_EXAMPLES:

        messages.append(
            {
                "role": "user",
                "content": f"""Solve the following math word problem.

Work through the problem step by step.

After your reasoning, give the final numerical answer in this exact format:

Final Answer: <number>

Problem:
{example["question"]}"""
            }
        )

        messages.append(
            {
                "role": "assistant",
                "content": prepare_training_answer(
                    example["answer"]
                )
            }
        )


    # Add the actual test question
    messages.append(
        {
            "role": "user",
            "content": f"""Solve the following math word problem.

Work through the problem step by step.

After your reasoning, give the final numerical answer in this exact format:

Final Answer: <number>

Problem:
{question}"""
        }
    )

    return messages


print("Few-shot prompt builder ready.")

Few-shot prompt builder ready.


Generation function

In [ ]:
# ============================================================
# FEW-SHOT CoT GENERATION
# ============================================================

def generate_fewshot_response(question):

    messages = build_fewshot_messages(question)

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=768,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode ONLY newly generated tokens
    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response


print("Few-shot generation function ready.")

Few-shot generation function ready.


Question 1-5 test

In [ ]:
# ============================================================
# FEW-SHOT CoT — TEST QUESTIONS 1–5
# ============================================================

fewshot_test_results = []


for _, row in df.head(5).iterrows():

    response = generate_fewshot_response(
        row["question"]
    )

    model_answer = extract_final_answer(
        response
    )

    correct_answer = normalize_answer(
        row["correct_answer"]
    )

    is_correct = (
        model_answer is not None
        and model_answer == correct_answer
    )

    fewshot_test_results.append(
        {
            "id": row["id"],
            "model_answer": model_answer,
            "correct_answer": correct_answer,
            "is_correct": is_correct,
            "raw_response": response
        }
    )

    print("=" * 70)

    print("QUESTION:", row["id"])

    print("Model answer:", model_answer)

    print("Correct answer:", correct_answer)

    print("Correct:", is_correct)

    print("\nRAW RESPONSE:")
    print(response)

    print()


fewshot_test_df = pd.DataFrame(
    fewshot_test_results
)

display(
    fewshot_test_df[
        [
            "id",
            "model_answer",
            "correct_answer",
            "is_correct"
        ]
    ]
)

QUESTION: 1
Model answer: 109
Correct answer: 109
Correct: True

RAW RESPONSE:
To solve this problem, we will follow these steps:

1. **Define Variables:**
   Let \( D \) represent Darrell's current age and \( A \) represent Allen's current age.

2. **Set Up the Ratio Equation:**
   According to the problem, the ratio of Darrell's age to Allen's age is 7:11. This can be written as:
   \[
   \frac{D}{A} = \frac{7}{11}
   \]
   From this ratio, we can express \( D \) in terms of \( A \):
   \[
   D = \frac{7}{11}A
   \]

3. **Set Up the Total Age Equation:**
   The problem states that their total age is 162. Therefore, we have:
   \[
   D + A = 162
   \]

4. **Substitute the Expression for \( D \):**
   Substitute \( D = \frac{7}{11}A \) into the total age equation:
   \[
   \frac{7}{11}A + A = 162
   \]

5. **Combine Like Terms:**
   To combine the terms on the left side, find a common denominator (which is 11):
   \[
   \frac{7A}{11} + \frac{11A}{11} = 162
   \]
   \[
   \frac{18A}{11}

,id,model_answer,correct_answer,is_correct
0,1,109,109,True
1,2,75,89,False
2,3,13,13,True
3,4,5,5,True
4,5,0,25,False


In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE FOR CHECKPOINTS
# ============================================================

from google.colab import drive
import os

drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/GSM8K_Research"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("Output folder:", OUTPUT_DIR)
print("Google Drive checkpointing ready.")

Mounted at /content/drive
Output folder: /content/drive/MyDrive/GSM8K_Research
Google Drive checkpointing ready.


Few-shot COT 200

In [ ]:
# ============================================================
# FEW-SHOT CoT — OFFICIAL 200-QUESTION RUN
# WITH GOOGLE DRIVE CHECKPOINTING
# ============================================================

import os
import pandas as pd


# ------------------------------------------------------------
# FILE PATHS
# ------------------------------------------------------------

FEWSHOT_CHECKPOINT = os.path.join(
    OUTPUT_DIR,
    "few_shot_cot_checkpoint.csv"
)

FEWSHOT_FINAL = os.path.join(
    OUTPUT_DIR,
    "few_shot_cot_FINAL_200.csv"
)


# ------------------------------------------------------------
# RESUME FROM CHECKPOINT IF ONE EXISTS
# ------------------------------------------------------------

if os.path.exists(FEWSHOT_CHECKPOINT):

    checkpoint_df = pd.read_csv(
        FEWSHOT_CHECKPOINT
    )

    fewshot_results = checkpoint_df.to_dict(
        "records"
    )

    completed_ids = set(
        checkpoint_df["id"]
        .astype(str)
        .tolist()
    )

    print(
        f"Checkpoint found. "
        f"Resuming after {len(fewshot_results)} completed questions."
    )

else:

    fewshot_results = []
    completed_ids = set()

    print("No checkpoint found.")
    print("Starting Few-shot CoT from Question 1.")


# ------------------------------------------------------------
# RUN ALL 200 QUESTIONS
# ------------------------------------------------------------

for _, row in df.iterrows():

    question_id = str(row["id"])

    # Skip questions already completed
    if question_id in completed_ids:
        continue


    # Generate model response
    response = generate_fewshot_response(
        row["question"]
    )


    # Extract model's final answer
    model_answer = extract_final_answer(
        response
    )


    # Normalize ground-truth answer
    correct_answer = normalize_answer(
        row["correct_answer"]
    )


    # Determine correctness
    is_correct = bool(
        model_answer is not None
        and model_answer == correct_answer
    )


    # Record extraction failure
    extraction_failed = (
        model_answer is None
    )


    # --------------------------------------------------------
    # SAVE EVERYTHING REQUIRED
    # --------------------------------------------------------

    result = {

        "id":
            row["id"],

        "question":
            row["question"],

        "model_response":
            response,

        "model_answer":
            model_answer,

        "correct_answer":
            correct_answer,

        "is_correct":
            is_correct,

        "extraction_failed":
            extraction_failed
    }


    fewshot_results.append(result)

    completed_ids.add(question_id)


    # --------------------------------------------------------
    # CHECKPOINT AFTER EVERY QUESTION
    # --------------------------------------------------------

    pd.DataFrame(
        fewshot_results
    ).to_csv(
        FEWSHOT_CHECKPOINT,
        index=False
    )


    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    print(
        f"[{len(fewshot_results)}/200] "
        f"ID={row['id']} | "
        f"model={model_answer} | "
        f"correct={correct_answer} | "
        f"match={is_correct} | "
        f"extraction_failed={extraction_failed}"
    )


# ------------------------------------------------------------
# CREATE FINAL DATAFRAME
# ------------------------------------------------------------

fewshot_final_df = pd.DataFrame(
    fewshot_results
)


# Make sure results are in original question order
order_map = {
    str(question_id): position
    for position, question_id
    in enumerate(df["id"].astype(str))
}

fewshot_final_df["_order"] = (
    fewshot_final_df["id"]
    .astype(str)
    .map(order_map)
)

fewshot_final_df = (
    fewshot_final_df
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

assert len(fewshot_final_df) == 200, (
    f"ERROR: Expected 200 results, "
    f"but found {len(fewshot_final_df)}."
)


# ------------------------------------------------------------
# SAVE FINAL REQUIRED CSV
# ------------------------------------------------------------

fewshot_final_df.to_csv(
    FEWSHOT_FINAL,
    index=False
)


# ------------------------------------------------------------
# FINAL STATISTICS
# ------------------------------------------------------------

fewshot_correct = int(
    fewshot_final_df[
        "is_correct"
    ].sum()
)


fewshot_extraction_failures = int(
    fewshot_final_df[
        "extraction_failed"
    ].sum()
)


fewshot_accuracy = (
    fewshot_correct / 200
)


print("\n" + "=" * 70)

print("FEW-SHOT CoT — FINAL RESULTS")

print("=" * 70)

print(
    f"Questions completed: "
    f"{len(fewshot_final_df)}/200"
)

print(
    f"Correct: "
    f"{fewshot_correct}/200"
)

print(
    f"Accuracy: "
    f"{fewshot_accuracy:.2%}"
)

print(
    f"Extraction failures: "
    f"{fewshot_extraction_failures}"
)

print("\nFinal CSV saved to:")

print(FEWSHOT_FINAL)

print("\nCheckpoint saved to:")

print(FEWSHOT_CHECKPOINT)

No checkpoint found.
Starting Few-shot CoT from Question 1.
[1/200] ID=1 | model=109 | correct=109 | match=True | extraction_failed=False
[2/200] ID=2 | model=75 | correct=89 | match=False | extraction_failed=False
[3/200] ID=3 | model=13 | correct=13 | match=True | extraction_failed=False
[4/200] ID=4 | model=5 | correct=5 | match=True | extraction_failed=False
[5/200] ID=5 | model=0 | correct=25 | match=False | extraction_failed=False
[6/200] ID=6 | model=692 | correct=452 | match=False | extraction_failed=False
[7/200] ID=7 | model=43 | correct=43 | match=True | extraction_failed=False
[8/200] ID=8 | model=34 | correct=34 | match=True | extraction_failed=False
[9/200] ID=9 | model=120 | correct=120 | match=True | extraction_failed=False
[10/200] ID=10 | model=11 | correct=11 | match=True | extraction_failed=False
[11/200] ID=11 | model=27 | correct=34 | match=False | extraction_failed=False
[12/200] ID=12 | model=12 | correct=12 | match=True | extraction_failed=False
[13/200] ID=13 

In [ ]:
# ============================================================
# FINAL FEW-SHOT CoT RESULTS — TEAM FORMAT
# ============================================================

fewshot_correct = int(
    fewshot_final_df["is_correct"].sum()
)

fewshot_accuracy = (
    fewshot_correct / 200
) * 100

failure_ids = fewshot_final_df.loc[
    fewshot_final_df["extraction_failed"] == True,
    "id"
].tolist()


print("=" * 35)
print("FINAL FEW-SHOT CoT RESULTS")
print("=" * 35)

print("Questions: 200")
print(f"Correct: {fewshot_correct}/200")
print(f"Accuracy: {fewshot_accuracy:.2f}%")
print(
    f"Remaining extraction failures: "
    f"{len(failure_ids)}"
)

print("\nRemaining failure IDs:")
print(failure_ids)

print("\nSaved:", FEWSHOT_FINAL)

FINAL FEW-SHOT CoT RESULTS
Questions: 200
Correct: 169/200
Accuracy: 84.50%
Remaining extraction failures: 2

Remaining failure IDs:
[83, 179]

Saved: /content/drive/MyDrive/GSM8K_Research/few_shot_cot_FINAL_200.csv


Self Consitency

In [ ]:
# ============================================================
# SELF-CONSISTENCY — SETUP
# ============================================================

from collections import Counter
import os
import pandas as pd
import torch


# Verify experiment setup
assert MODEL_NAME == "Qwen/Qwen2.5-3B-Instruct"
assert len(df) == 200


# Exact generation settings
SC_NUM_SAMPLES = 5
SC_TEMPERATURE = 0.7
SC_TOP_P = 0.95
SC_MAX_NEW_TOKENS = 768


print("Model:", MODEL_NAME)
print("Questions:", len(df))
print("Samples per question:", SC_NUM_SAMPLES)
print("Temperature:", SC_TEMPERATURE)
print("Top-p:", SC_TOP_P)
print("Max new tokens:", SC_MAX_NEW_TOKENS)

Model: Qwen/Qwen2.5-3B-Instruct
Questions: 200
Samples per question: 5
Temperature: 0.7
Top-p: 0.95
Max new tokens: 768


Generation

In [ ]:
# ============================================================
# SELF-CONSISTENCY — PROMPT + GENERATION
# ============================================================

def build_sc_messages(question):

    return [
        {
            "role": "user",
            "content": f"""Solve the following math word problem.

Work through the problem step by step.

After your reasoning, give the final numerical answer in this exact format:

Final Answer: <number>

Problem:
{question}"""
        }
    ]


def generate_sc_response(question):

    messages = build_sc_messages(question)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=SC_MAX_NEW_TOKENS,

            # Sampling is REQUIRED for Self-Consistency
            do_sample=True,
            temperature=SC_TEMPERATURE,
            top_p=SC_TOP_P,

            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response


print("Self-Consistency generation function ready.")

Self-Consistency generation function ready.


Majority vote function

In [ ]:
# ============================================================
# SELF-CONSISTENCY — MAJORITY VOTE
# ============================================================

def select_mode_answer(answers):

    # Remove extraction failures from voting
    valid_answers = [
        answer
        for answer in answers
        if answer is not None
    ]

    # All five failed extraction
    if len(valid_answers) == 0:
        return None, False


    counts = Counter(valid_answers)

    highest_count = max(counts.values())

    winners = [
        answer
        for answer, count in counts.items()
        if count == highest_count
    ]


    # Record whether there was a tie
    mode_tie = len(winners) > 1


    # Deterministic tie-break:
    # first occurring tied answer wins
    for answer in valid_answers:

        if answer in winners:
            return answer, mode_tie


print("Self-Consistency voting function ready.")

Self-Consistency voting function ready.


5 Test questions

In [ ]:
# ============================================================
# SELF-CONSISTENCY — ONE QUESTION FUNCTION
# ============================================================

def run_self_consistency_question(row):

    responses = []
    answers = []


    # Generate 5 independent sampled responses
    for sample_number in range(SC_NUM_SAMPLES):

        response = generate_sc_response(
            row["question"]
        )

        answer = extract_final_answer(
            response
        )

        responses.append(response)
        answers.append(answer)


    # Majority/mode vote
    selected_answer, mode_tie = (
        select_mode_answer(answers)
    )


    correct_answer = normalize_answer(
        row["correct_answer"]
    )


    is_correct = bool(
        selected_answer is not None
        and selected_answer == correct_answer
    )


    num_extraction_failures = sum(
        answer is None
        for answer in answers
    )


    result = {

        "id":
            row["id"],

        "question":
            row["question"],

        "self_consistency_answer":
            selected_answer,

        "correct_answer":
            correct_answer,

        "is_correct":
            is_correct,

        "num_extraction_failures":
            num_extraction_failures,

        "all_extractions_failed":
            num_extraction_failures == SC_NUM_SAMPLES,

        "mode_tie":
            mode_tie
    }


    # Save ALL five raw responses and extracted answers
    for i in range(SC_NUM_SAMPLES):

        result[f"response_{i+1}"] = responses[i]

        result[f"answer_{i+1}"] = answers[i]


    return result


print("Self-Consistency question function ready.")

Self-Consistency question function ready.


In [ ]:
# ============================================================
# ANSWER NORMALIZATION + EXTRACTION
# Required again because Colab runtime restarted
# ============================================================

import re
import math
import pandas as pd


def normalize_answer(answer):

    if answer is None:
        return None

    if isinstance(answer, float) and math.isnan(answer):
        return None

    answer = str(answer).strip()
    answer = answer.replace(",", "")
    answer = answer.replace("$", "")

    try:
        number = float(answer)

        if number.is_integer():
            return str(int(number))

        return str(number)

    except (ValueError, TypeError):
        return answer if answer else None


def extract_final_answer(response):

    if response is None:
        return None

    if pd.isna(response):
        return None

    response = str(response)

    # Preferred format: Final Answer: <number>
    final_answer = re.findall(
        r"Final\s+Answer\s*:\s*"
        r"(?:\\\(\s*)?"
        r"\\?\$?\s*"
        r"(-?[\d,]+(?:\.\d+)?)",
        response,
        re.IGNORECASE
    )

    if final_answer:
        return normalize_answer(final_answer[-1])

    # Accept boxed numerical answer
    boxed = re.findall(
        r"\\boxed\{\s*\\?\$?\s*(-?[\d,]+(?:\.\d+)?)\s*\}",
        response
    )

    if boxed:
        return normalize_answer(boxed[-1])

    # Genuine extraction failure
    return None


print("Answer normalization and extraction functions restored.")

Answer normalization and extraction functions restored.


In [ ]:
# ============================================================
# SELF-CONSISTENCY — TEST QUESTIONS 1–5
# ============================================================

sc_test_results = []


for _, row in df.head(5).iterrows():

    print("\n" + "=" * 70)
    print("Running Question:", row["id"])

    result = run_self_consistency_question(row)

    sc_test_results.append(result)


    answers = [
        result[f"answer_{i}"]
        for i in range(1, 6)
    ]


    print("Five extracted answers:", answers)

    print(
        "Selected answer:",
        result["self_consistency_answer"]
    )

    print(
        "Correct answer:",
        result["correct_answer"]
    )

    print(
        "Correct:",
        result["is_correct"]
    )

    print(
        "Extraction failures:",
        result["num_extraction_failures"]
    )

    print(
        "Mode tie:",
        result["mode_tie"]
    )


# Create test dataframe
sc_test_df = pd.DataFrame(
    sc_test_results
)


print("\n" + "=" * 70)
print("SELF-CONSISTENCY — Q1–5 TEST RESULTS")
print("=" * 70)


display(
    sc_test_df[
        [
            "id",
            "answer_1",
            "answer_2",
            "answer_3",
            "answer_4",
            "answer_5",
            "self_consistency_answer",
            "correct_answer",
            "is_correct",
            "num_extraction_failures",
            "mode_tie"
        ]
    ]
)


Running Question: 1
Five extracted answers: ['109', '109', '109', '109', '109']
Selected answer: 109
Correct answer: 109
Correct: True
Extraction failures: 0
Mode tie: False

Running Question: 2
Five extracted answers: ['12', '30', '31', '29', '30']
Selected answer: 30
Correct answer: 89
Correct: False
Extraction failures: 0
Mode tie: False

Running Question: 3
Five extracted answers: ['13', None, '13', '13', '13']
Selected answer: 13
Correct answer: 13
Correct: True
Extraction failures: 1
Mode tie: False

Running Question: 4
Five extracted answers: ['5', '5', '5', '5', '5']
Selected answer: 5
Correct answer: 5
Correct: True
Extraction failures: 0
Mode tie: False

Running Question: 5
Five extracted answers: ['25', '40', '0', '25', '25']
Selected answer: 25
Correct answer: 25
Correct: True
Extraction failures: 0
Mode tie: False

SELF-CONSISTENCY — Q1–5 TEST RESULTS


,id,answer_1,answer_2,answer_3,answer_4,answer_5,self_consistency_answer,correct_answer,is_correct,num_extraction_failures,mode_tie
0,1,109,109,109,109,109,109,109,True,0,False
1,2,12,30,31,29,30,30,89,False,0,False
2,3,13,None,13,13,13,13,13,True,1,False
3,4,5,5,5,5,5,5,5,True,0,False
4,5,25,40,0,25,25,25,25,True,0,False


google drive remount

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/GSM8K_Research"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output folder:", OUTPUT_DIR)
print("Google Drive checkpointing ready.")

Mounted at /content/drive
Output folder: /content/drive/MyDrive/GSM8K_Research
Google Drive checkpointing ready.


200 questions

In [ ]:
# ============================================================
# SELF-CONSISTENCY — OFFICIAL 200-QUESTION RUN
# 5 SAMPLED RESPONSES PER QUESTION
# WITH GOOGLE DRIVE CHECKPOINTING
# ============================================================

import os
import pandas as pd


# ------------------------------------------------------------
# FILE PATHS
# ------------------------------------------------------------

SC_CHECKPOINT = os.path.join(
    OUTPUT_DIR,
    "self_consistency_checkpoint.csv"
)

SC_FINAL = os.path.join(
    OUTPUT_DIR,
    "self_consistency_FINAL_200.csv"
)


# ------------------------------------------------------------
# RESUME FROM CHECKPOINT IF IT EXISTS
# ------------------------------------------------------------

if os.path.exists(SC_CHECKPOINT):

    checkpoint_df = pd.read_csv(
        SC_CHECKPOINT
    )

    sc_results = checkpoint_df.to_dict(
        "records"
    )

    completed_ids = set(
        checkpoint_df["id"]
        .astype(str)
        .tolist()
    )

    print(
        f"Checkpoint found. "
        f"Resuming after {len(sc_results)} "
        f"completed questions."
    )

else:

    sc_results = []
    completed_ids = set()

    print("No checkpoint found.")
    print(
        "Starting Self-Consistency "
        "from Question 1."
    )


# ------------------------------------------------------------
# RUN ALL 200 QUESTIONS
# ------------------------------------------------------------

for _, row in df.iterrows():

    question_id = str(row["id"])

    # Skip already completed questions
    if question_id in completed_ids:
        continue


    print(
        f"\nRunning question "
        f"{len(sc_results) + 1}/200 "
        f"(ID={row['id']})..."
    )


    # --------------------------------------------------------
    # Generate 5 responses + vote
    # --------------------------------------------------------

    result = run_self_consistency_question(
        row
    )


    sc_results.append(result)

    completed_ids.add(question_id)


    # --------------------------------------------------------
    # CHECKPOINT AFTER EVERY QUESTION
    # --------------------------------------------------------

    pd.DataFrame(
        sc_results
    ).to_csv(
        SC_CHECKPOINT,
        index=False
    )


    # --------------------------------------------------------
    # SHOW PROGRESS
    # --------------------------------------------------------

    five_answers = [
        result[f"answer_{i}"]
        for i in range(1, 6)
    ]


    print(
        "Five answers:",
        five_answers
    )

    print(
        f"[{len(sc_results)}/200] "
        f"selected="
        f"{result['self_consistency_answer']} | "
        f"correct="
        f"{result['correct_answer']} | "
        f"match="
        f"{result['is_correct']} | "
        f"extraction_failures="
        f"{result['num_extraction_failures']} | "
        f"mode_tie="
        f"{result['mode_tie']}"
    )


# ------------------------------------------------------------
# CREATE FINAL DATAFRAME
# ------------------------------------------------------------

sc_final_df = pd.DataFrame(
    sc_results
)


# ------------------------------------------------------------
# RESTORE ORIGINAL QUESTION ORDER
# ------------------------------------------------------------

order_map = {
    str(question_id): position
    for position, question_id
    in enumerate(df["id"].astype(str))
}


sc_final_df["_order"] = (
    sc_final_df["id"]
    .astype(str)
    .map(order_map)
)


sc_final_df = (
    sc_final_df
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

assert len(sc_final_df) == 200, (
    f"ERROR: Expected 200 results, "
    f"but found {len(sc_final_df)}."
)


# ------------------------------------------------------------
# SAVE FINAL REQUIRED CSV
# ------------------------------------------------------------

sc_final_df.to_csv(
    SC_FINAL,
    index=False
)


# ------------------------------------------------------------
# FINAL STATISTICS
# ------------------------------------------------------------

sc_correct = int(
    sc_final_df[
        "is_correct"
    ].sum()
)


sc_accuracy = (
    sc_correct / 200
) * 100


# Total failed extractions among all 1000 generations
total_sample_extraction_failures = int(
    sc_final_df[
        "num_extraction_failures"
    ].sum()
)


# Questions where ALL 5 extractions failed
all_failed_ids = (
    sc_final_df.loc[
        sc_final_df[
            "all_extractions_failed"
        ] == True,
        "id"
    ]
    .tolist()
)


# Questions where mode voting had a tie
mode_tie_ids = (
    sc_final_df.loc[
        sc_final_df[
            "mode_tie"
        ] == True,
        "id"
    ]
    .tolist()
)


# ------------------------------------------------------------
# TEAM-STYLE FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 35)

print(
    "FINAL SELF-CONSISTENCY RESULTS"
)

print("=" * 35)

print("Questions: 200")

print(
    f"Correct: {sc_correct}/200"
)

print(
    f"Accuracy: {sc_accuracy:.2f}%"
)

print(
    "Sample extraction failures: "
    f"{total_sample_extraction_failures}/1000"
)

print(
    "Questions with all 5 "
    "extractions failed: "
    f"{len(all_failed_ids)}"
)

print("\nAll-failed question IDs:")
print(all_failed_ids)

print(
    "\nMode-tie question IDs:"
)
print(mode_tie_ids)

print(
    "\nSaved:",
    SC_FINAL
)

No checkpoint found.
Starting Self-Consistency from Question 1.

Running question 1/200 (ID=1)...
Five answers: ['109', '109', '109', '109', '109']
[1/200] selected=109 | correct=109 | match=True | extraction_failures=0 | mode_tie=False

Running question 2/200 (ID=2)...
Five answers: ['33', '29', '94', '41', '72']
[2/200] selected=33 | correct=89 | match=False | extraction_failures=0 | mode_tie=True

Running question 3/200 (ID=3)...
Five answers: ['13', '13', '13', '13', '13']
[3/200] selected=13 | correct=13 | match=True | extraction_failures=0 | mode_tie=False

Running question 4/200 (ID=4)...
Five answers: ['5', '5', '5', None, '5']
[4/200] selected=5 | correct=5 | match=True | extraction_failures=1 | mode_tie=False

Running question 5/200 (ID=5)...
Five answers: ['25', '0', '25', '30', '0']
[5/200] selected=25 | correct=25 | match=True | extraction_failures=0 | mode_tie=True

Running question 6/200 (ID=6)...
Five answers: ['697', '452', '447', '452', None]
[6/200] selected=452 | co

In [ ]:
!nvidia-smi

Wed Sep 23 15:12:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers accelerate datasets

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded:", MODEL_NAME)
print("Device:", model.device)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-3B-Instruct
Device: cuda:0


In [ ]:
import os

print("Checkpoint:", os.path.exists("/content/self_consistency_checkpoint.csv"))
print("Original dataset:", os.path.exists("/content/gsm8k_200_questions.csv"))

Checkpoint: True
Original dataset: True


In [ ]:
import pandas as pd

df = pd.read_csv("/content/gsm8k_200_questions.csv")
checkpoint = pd.read_csv("/content/self_consistency_checkpoint.csv")

print("Original questions:", len(df))
print("Checkpoint questions:", len(checkpoint))
print("Last saved ID:", checkpoint.iloc[-1]["id"])

print("\nFirst 5 correct answers:")
print(df["correct_answer"].head().tolist())

Original questions: 200
Checkpoint questions: 192
Last saved ID: 192

First 5 correct answers:
[109, 89, 13, 5, 25]


In [ ]:
import re
import math
import pandas as pd

def normalize_answer(answer):
    if answer is None:
        return None

    if isinstance(answer, float) and math.isnan(answer):
        return None

    answer = str(answer).strip()
    answer = answer.replace(",", "")
    answer = answer.replace("$", "")

    try:
        number = float(answer)

        if number.is_integer():
            return str(int(number))

        return str(number)

    except (ValueError, TypeError):
        return answer if answer else None


def extract_final_answer(response):
    if response is None:
        return None

    if pd.isna(response):
        return None

    response = str(response)

    final_answer = re.findall(
        r"Final\s+Answer\s*:\s*"
        r"(?:\\\(\s*)?"
        r"\\?\$?\s*"
        r"(-?[\d,]+(?:\.\d+)?)",
        response,
        re.IGNORECASE
    )

    if final_answer:
        return normalize_answer(final_answer[-1])

    boxed = re.findall(
        r"\\boxed\{\s*\\?\$?\s*(-?[\d,]+(?:\.\d+)?)\s*\}",
        response
    )

    if boxed:
        return normalize_answer(boxed[-1])

    return None


print("Answer extractor restored.")

Answer extractor restored.


In [ ]:
from collections import Counter
import torch

assert MODEL_NAME == "Qwen/Qwen2.5-3B-Instruct"
assert len(df) == 200

SC_NUM_SAMPLES = 5
SC_TEMPERATURE = 0.7
SC_TOP_P = 0.95
SC_MAX_NEW_TOKENS = 768

print("Model:", MODEL_NAME)
print("Questions:", len(df))
print("Samples per question:", SC_NUM_SAMPLES)
print("Temperature:", SC_TEMPERATURE)
print("Top-p:", SC_TOP_P)
print("Max new tokens:", SC_MAX_NEW_TOKENS)

Model: Qwen/Qwen2.5-3B-Instruct
Questions: 200
Samples per question: 5
Temperature: 0.7
Top-p: 0.95
Max new tokens: 768


In [ ]:
def build_sc_messages(question):
    return [
        {
            "role": "user",
            "content": f"""Solve the following math word problem.

Work through the problem step by step.

After your reasoning, give the final numerical answer in this exact format:

Final Answer: <number>

Problem:
{question}"""
        }
    ]


def generate_sc_response(question):
    messages = build_sc_messages(question)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=SC_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=SC_TEMPERATURE,
            top_p=SC_TOP_P,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response


print("Self-Consistency generation function ready.")

Self-Consistency generation function ready.


In [ ]:
def select_mode_answer(answers):
    valid_answers = [
        answer
        for answer in answers
        if answer is not None
    ]

    if len(valid_answers) == 0:
        return None, False

    counts = Counter(valid_answers)
    highest_count = max(counts.values())

    winners = [
        answer
        for answer, count in counts.items()
        if count == highest_count
    ]

    mode_tie = len(winners) > 1

    # If tied, select the earliest occurring tied answer
    for answer in valid_answers:
        if answer in winners:
            return answer, mode_tie


def run_self_consistency_question(row):
    responses = []
    answers = []

    for sample_number in range(SC_NUM_SAMPLES):
        response = generate_sc_response(
            row["question"]
        )

        answer = extract_final_answer(
            response
        )

        responses.append(response)
        answers.append(answer)

    selected_answer, mode_tie = select_mode_answer(
        answers
    )

    correct_answer = normalize_answer(
        row["correct_answer"]
    )

    is_correct = bool(
        selected_answer is not None
        and selected_answer == correct_answer
    )

    num_extraction_failures = sum(
        answer is None
        for answer in answers
    )

    result = {
        "id": row["id"],
        "question": row["question"],
        "self_consistency_answer": selected_answer,
        "correct_answer": correct_answer,
        "is_correct": is_correct,
        "num_extraction_failures": num_extraction_failures,
        "all_extractions_failed":
            num_extraction_failures == SC_NUM_SAMPLES,
        "mode_tie": mode_tie
    }

    for i in range(SC_NUM_SAMPLES):
        result[f"response_{i+1}"] = responses[i]
        result[f"answer_{i+1}"] = answers[i]

    return result


print("Self-Consistency voting and question functions ready.")

Self-Consistency voting and question functions ready.


In [ ]:
import pandas as pd
import os

SC_CHECKPOINT = "/content/self_consistency_checkpoint.csv"
SC_FINAL = "/content/self_consistency_FINAL_200.csv"

# Load the 192 completed results
checkpoint_df = pd.read_csv(SC_CHECKPOINT)

sc_results = checkpoint_df.to_dict("records")
completed_ids = set(
    checkpoint_df["id"].astype(str)
)

print(
    f"Checkpoint found. "
    f"Resuming after {len(sc_results)} completed questions."
)

print("Remaining questions:",
      len(df) - len(sc_results))

# Run ONLY questions not already in checkpoint
for _, row in df.iterrows():

    qid = str(row["id"])

    if qid in completed_ids:
        continue

    print(
        f"\nRunning question {row['id']}/200..."
    )

    result = run_self_consistency_question(row)

    sc_results.append(result)
    completed_ids.add(qid)

    # Save immediately after every completed question
    temp_df = pd.DataFrame(sc_results)
    temp_df.to_csv(
        SC_CHECKPOINT,
        index=False
    )

    print(
        "Five answers:",
        [
            result[f"answer_{i}"]
            for i in range(1, 6)
        ]
    )
    print(
        "Selected:",
        result["self_consistency_answer"]
    )
    print(
        "Correct answer:",
        result["correct_answer"]
    )
    print(
        "Is correct:",
        result["is_correct"]
    )
    print(
        f"Checkpoint saved: "
        f"{len(sc_results)}/200"
    )

print("\nGeneration complete.")
print("Total saved:", len(sc_results))

Checkpoint found. Resuming after 192 completed questions.
Remaining questions: 8

Running question 193/200...
Five answers: ['55', '55', '55', '55', '55']
Selected: 55
Correct answer: 55
Is correct: True
Checkpoint saved: 193/200

Running question 194/200...
Five answers: ['490', '490', '490', '490', '490']
Selected: 490
Correct answer: 490
Is correct: True
Checkpoint saved: 194/200

Running question 195/200...
Five answers: ['20', '20', '20', '40', '20']
Selected: 20
Correct answer: 20
Is correct: True
Checkpoint saved: 195/200

Running question 196/200...
Five answers: ['81', '120', '81', '120', '363']
Selected: 81
Correct answer: 81
Is correct: True
Checkpoint saved: 196/200

Running question 197/200...
Five answers: ['250', None, '250', '250', '250']
Selected: 250
Correct answer: 250
Is correct: True
Checkpoint saved: 197/200

Running question 198/200...
Five answers: ['800', '800', '800', '800', '800']
Selected: 800
Correct answer: 800
Is correct: True
Checkpoint saved: 198/200

R

In [ ]:
import pandas as pd

SC_CHECKPOINT = "/content/self_consistency_checkpoint.csv"
SC_FINAL = "/content/self_consistency_FINAL_200.csv"

# Load completed 200-question checkpoint
sc_final_df = pd.read_csv(SC_CHECKPOINT)

# Safety checks
assert len(sc_final_df) == 200, \
    f"Expected 200 questions, found {len(sc_final_df)}"

assert sc_final_df["id"].nunique() == 200, \
    "Duplicate question IDs found!"

# Preserve original dataset order
id_order = {
    str(qid): i
    for i, qid in enumerate(df["id"])
}

sc_final_df["_order"] = (
    sc_final_df["id"]
    .astype(str)
    .map(id_order)
)

sc_final_df = (
    sc_final_df
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

# Normalize columns loaded from CSV
sc_final_df["is_correct"] = (
    sc_final_df["is_correct"]
    .astype(str)
    .str.lower()
    .eq("true")
)

sc_final_df["all_extractions_failed"] = (
    sc_final_df["all_extractions_failed"]
    .astype(str)
    .str.lower()
    .eq("true")
)

sc_final_df["mode_tie"] = (
    sc_final_df["mode_tie"]
    .astype(str)
    .str.lower()
    .eq("true")
)

sc_final_df["num_extraction_failures"] = (
    pd.to_numeric(
        sc_final_df["num_extraction_failures"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

# Calculate final statistics
correct = int(sc_final_df["is_correct"].sum())
accuracy = correct / 200 * 100

sample_failures = int(
    sc_final_df["num_extraction_failures"].sum()
)

all_failed_ids = sc_final_df.loc[
    sc_final_df["all_extractions_failed"],
    "id"
].tolist()

mode_tie_ids = sc_final_df.loc[
    sc_final_df["mode_tie"],
    "id"
].tolist()

# Save FINAL deliverable
sc_final_df.to_csv(
    SC_FINAL,
    index=False
)

print("===================================")
print("FINAL SELF-CONSISTENCY RESULTS")
print("===================================")
print("Questions: 200")
print(f"Correct: {correct}/200")
print(f"Accuracy: {accuracy:.2f}%")
print(
    f"Sample extraction failures: "
    f"{sample_failures}/1000"
)
print(
    "Questions with all 5 extractions failed:",
    len(all_failed_ids)
)

print("\nAll-failed question IDs:")
print(all_failed_ids)

print("\nMode-tie question IDs:")
print(mode_tie_ids)

print("\nSaved:", SC_FINAL)

FINAL SELF-CONSISTENCY RESULTS
Questions: 200
Correct: 181/200
Accuracy: 90.50%
Sample extraction failures: 68/1000
Questions with all 5 extractions failed: 0

All-failed question IDs:
[]

Mode-tie question IDs:
[2, 5, 58, 70, 71, 89, 115, 158, 196]

Saved: /content/self_consistency_FINAL_200.csv
